In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.coordinates as coord
import astropy.units as u
import pandas as pd
import sys
from scipy.special import factorial
from astropy.table import Table
from numpy.polynomial.polynomial import Polynomial
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib
from sklearn.decomposition import FastICA
from sklearn.feature_selection import mutual_info_regression
from sklearn.decomposition import PCA
from scipy.stats import ks_2samp
import seaborn as sns

if './SelfCalGroupFinder/py/' not in sys.path:
    sys.path.append('./SelfCalGroupFinder/py/')
from pyutils import *
import plotting as pp
from dataloc import *
from bgs_helpers import *
import catalog_definitions as cat
from groupcatalog import *
from halosimutils import *

%load_ext autoreload
%autoreload 2

In [ ]:
BOX_SIZE_MPC_H = 400  # Mpc/h, from the smdpl documentation
BOX_VOLUME = BOX_SIZE_MPC_H**3  # (Mpc/h)^3

feature_cols = ['LOGMHALO', 'Halfmass_Scale', 'c', 'Spin', ]
#feature_cols = ['LOGMHALO', 'Halfmass_Scale']
n_features = len(feature_cols) # No dimensionality reduction right now. 

# Define file paths
directory = '/mount/sirocco1/imw2293/GROUP_CAT/DATA/POPMOCK/'
file = directory + 'smdpl_z0.19717.M1E11.C.h5' # for latsham this is what we want for now. For group catalog 8e9 is needed.

In [ ]:
# Read the entire HDF5 file into memory
df = pd.read_hdf(file, key='halos')

# I think rvir is OK for this c calculation
#df['c'] = df['rvir'] / df['rs'] # both in kpc/h, so this is dimensionless
#df['LOGMHALO'] = np.log10(df['M200b']) # Mpeak instead of M200b? Sometimes it way higher for some reason. Keep M200b

In [ ]:
df_down = downsample_halos(df)

# Plot orginal vs downsampled mass distribution
plt.figure(figsize=(6, 4))
plt.hist(df['LOGMHALO'], histtype='step', bins=50, label='Original', color='blue')
plt.hist(df_down['LOGMHALO'], histtype='step', bins=50, label='Downsampled', color='orange')
plt.yscale('log')
plt.xlabel('LOGMHALO')
plt.ylabel('Count')
plt.legend()

# And show their correlation matrices
plt.figure(figsize=(6, 5))
plt.subplot(1, 1, 1)
sns.heatmap(df[feature_cols].corr(), annot=True, cmap='coolwarm', vmin=-1, vmax=1, xticklabels=True, yticklabels=True)
# rename the axes to be more readable
plt.xticks(ticks=np.arange(len(feature_cols)) + 0.5, labels=['$\\log M_{\\rm h}$', 'c', '$\\lambda$', '$a_{1/2}$'], rotation=45, ha='right')
plt.yticks(ticks=np.arange(len(feature_cols)) + 0.5, labels=['$\\log M_{\\rm h}$', 'c', '$\\lambda$', '$a_{1/2}$'], rotation=0)
plt.title('Correlation Matrix of Original Data')
#plt.subplot(1, 2, 2)
#sns.heatmap(df_down[feature_cols].corr(), annot=True, cmap='coolwarm', vmin=-1, vmax=1, xticklabels=True, yticklabels=True)
#plt.title('Correlation Matrix of Downsampled Data') 

In [ ]:
# Drop rows with any NaN in the selected features

## !!!!
### Trying where we train on everything instead. Was going to use df_down here. TODO
train_clean = df.dropna(subset=feature_cols)
all_clean = df.dropna(subset=feature_cols)
print(f"Samples after NaN drop: {len(all_clean)} / {len(df)}")

In [ ]:
# Scale first - critical since features have very different units/ranges
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_clean[feature_cols])
all_scaled = scaler.transform(all_clean[feature_cols])

In [ ]:
# Fit full PCA on training set (where we've removed a lot of low mass halos)
ica = FastICA(n_components=n_features, whiten='unit-variance', random_state=687, max_iter=2000)
train_ica = ica.fit_transform(train_scaled)

# Reorder: sort ICs by abs correlation with LOGMHALO (descending)
# Fix sign: IC most correlated with LOGMHALO should have positive correlation
corr = np.corrcoef(train_ica.T, train_clean['LOGMHALO'].values)[:-1, -1]
order = np.argsort(-np.abs(corr))
signs = np.sign(corr[order])
# Apply permutation + sign flip to ica.components_ and ica.mixing_
ica.components_ = (ica.components_[order].T * signs).T
ica.mixing_ = (ica.mixing_.T[order].T) * signs  # update mixing accordingly

del train_ica # Done with this now that we've fixed the order and signs and have the model. This data doesn't thave order/signs fixed!

In [ ]:
# Print feature loadings for top components
print("\nFeature loadings:")
loadings = pd.DataFrame(
    ica.components_.T,
    index=feature_cols,
    columns=[f'IC{i+1}' for i in range(ica.components_.shape[0])]
)
print(loadings.round(3).to_string())

IC1 is high halo mass, almost entirly
IC2 is low concentration, and early forming
IC3 is early forming and high spin
IC4 is high spin, almost entirely

In [ ]:

# Fit PCA on same training data for comparison
pca = PCA(n_components=n_features)
pca.fit(all_scaled)

In [ ]:
# Print feature loadings for top components
print("\nFeature loadings:")
loadings = pd.DataFrame(
    pca.components_.T,
    index=feature_cols,
    columns=[f'PC{i+1}' for i in range(pca.components_.shape[0])]
)
print(loadings.round(3).to_string())

In [ ]:
# Apply ICA transformation and join the columns
reduced_ica = ica.transform(train_scaled)
reduced_pca = pca.transform(train_scaled)
reduced_with_ica = pd.concat([train_clean.reset_index(drop=True), pd.DataFrame(reduced_ica, columns=[f'ICA{i+1}' for i in range(reduced_ica.shape[1])])], axis=1)
reduced_sample = pd.concat([reduced_with_ica.reset_index(drop=True), pd.DataFrame(reduced_pca, columns=[f'PCA{i+1}' for i in range(reduced_pca.shape[1])])], axis=1)

all_ica_full = ica.transform(all_scaled)
all_pca_full = pca.transform(all_scaled)
all_with_ica = pd.concat([all_clean.reset_index(drop=True), pd.DataFrame(all_ica_full, columns=[f'ICA{i+1}' for i in range(all_ica_full.shape[1])])], axis=1)
complete_sample = pd.concat([all_with_ica.reset_index(drop=True), pd.DataFrame(all_pca_full, columns=[f'PCA{i+1}' for i in range(all_pca_full.shape[1])])], axis=1)

### What did ICA and PCA do?

In [ ]:
# Examine some contour plots of the a sample of objects in the original feature space, ICA space, and PCA space
sample_trained = reduced_sample.sample(5000, random_state=687)
sample = complete_sample.sample(5000, random_state=687)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))


sns.kdeplot(x=sample['LOGMHALO'], y=sample['Halfmass_Scale'], levels=10, cmap='Blues', ax=axes[0, 0])
sns.kdeplot(x=sample['ICA1'], y=sample['ICA2'], levels=10, cmap='Oranges', ax=axes[0, 1])
sns.kdeplot(x=sample['PCA1'], y=sample['PCA2'], levels=10, cmap='Greens', ax=axes[0, 2])
sns.kdeplot(x=sample_trained['LOGMHALO'], y=sample_trained['Halfmass_Scale'], levels=10, cmap='Blues', ax=axes[1, 0])
sns.kdeplot(x=sample_trained['ICA1'], y=sample_trained['ICA2'], levels=10, cmap='Oranges', ax=axes[1, 1])
sns.kdeplot(x=sample_trained['PCA1'], y=sample_trained['PCA2'], levels=10, cmap='Greens', ax=axes[1, 2])

x_min = min(sample['LOGMHALO'].min(), sample_trained['LOGMHALO'].min())
x_max = max(sample['LOGMHALO'].max(), sample_trained['LOGMHALO'].max())
y_min = min(sample['Halfmass_Scale'].min(), sample_trained['Halfmass_Scale'].min())
y_max = max(sample['Halfmass_Scale'].max(), sample_trained['Halfmass_Scale'].max())
axes[0, 0].set_xlim(x_min, x_max)
axes[0, 0].set_ylim(y_min, y_max)
axes[1, 0].set_xlim(x_min, x_max)
axes[1, 0].set_ylim(y_min, y_max)
x_min = min(sample['ICA1'].min(), sample_trained['ICA1'].min())
x_max = max(sample['ICA1'].max(), sample_trained['ICA1'].max())
y_min = min(sample['ICA2'].min(), sample_trained['ICA2'].min())
y_max = max(sample['ICA2'].max(), sample_trained['ICA2'].max())
axes[0, 1].set_xlim(x_min, x_max)
axes[0, 1].set_ylim(y_min, y_max)
axes[1, 1].set_xlim(x_min, x_max)
axes[1, 1].set_ylim(y_min, y_max)
x_min = min(sample['PCA1'].min(), sample_trained['PCA1'].min())
x_max = max(sample['PCA1'].max(), sample_trained['PCA1'].max())
y_min = min(sample['PCA2'].min(), sample_trained['PCA2'].min())
y_max = max(sample['PCA2'].max(), sample_trained['PCA2'].max())
axes[0, 2].set_xlim(x_min, x_max)
axes[0, 2].set_ylim(y_min, y_max)
axes[1, 2].set_xlim(x_min, x_max)
axes[1, 2].set_ylim(y_min, y_max)

axes[0, 0].set_title('Original Space (Full Sample)')
axes[0, 1].set_title('ICA Space (Full Sample)')
axes[0, 2].set_title('PCA Space (Full Sample)')
axes[1, 0].set_title('Original Space (Trained Sample)')
axes[1, 1].set_title('ICA Space (Trained Sample)')
axes[1, 2].set_title('PCA Space (Trained Sample)')
plt.tight_layout()
plt.show()

# Label

In [ ]:
# Examine some scatter plots of the a sample of objects in the original feature space, ICA space, and PCA space
sample_trained = reduced_sample.sample(3000, random_state=687)
sample = complete_sample.sample(3000, random_state=687)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
alpha = 0.2
size = 5

axes[0, 0].scatter(sample['LOGMHALO'], sample['Halfmass_Scale'], alpha=alpha, s=size)
axes[1, 0].scatter(sample_trained['LOGMHALO'], sample_trained['Halfmass_Scale'], alpha=alpha, s=size, color='red', label='Trained Sample')
axes[0, 0].set_xlabel('LOGMHALO')
axes[0, 0].set_ylabel('Halfmass_Scale')
axes[0, 0].set_title('Original Feature Space')
axes[1, 0].set_xlabel('LOGMHALO')
axes[1, 0].set_ylabel('Halfmass_Scale')
axes[0, 1].scatter(sample['ICA1'], sample['ICA2'], alpha=alpha, s=size)
axes[1, 1].scatter(sample_trained['ICA1'], sample_trained['ICA2'], alpha=alpha, s=size, color='red', label='Trained Sample')
axes[0, 1].set_xlabel('ICA1')
axes[0, 1].set_ylabel('ICA2')
axes[1, 1].set_xlabel('ICA1')
axes[1, 1].set_ylabel('ICA2')
axes[0, 1].set_title('ICA Space')
axes[0, 2].scatter(sample['PCA1'], sample['PCA2'], alpha=alpha, s=size)
axes[1, 2].scatter(sample_trained['PCA1'], sample_trained['PCA2'], alpha=alpha, s=size, color='red', label='Trained Sample')
axes[0, 2].set_xlabel('PCA1')
axes[0, 2].set_ylabel('PCA2')
axes[1, 2].set_xlabel('PCA1')
axes[1, 2].set_ylabel('PCA2')
axes[0, 2].set_title('PCA Space')
plt.tight_layout()
plt.show()

In [ ]:
# First let's see if correlation matrix is now diagonal for both
corr_ica = pd.DataFrame(all_ica_full).corr()
corr_pca = pd.DataFrame(all_pca_full).corr()
corr_orig = pd.DataFrame(all_clean[feature_cols]).corr()

# Show
plt.figure(figsize=(18, 5))
plt.subplot(1, 3, 1)
sns.heatmap(corr_ica, annot=True, cmap='coolwarm', vmin=-1, vmax=1, xticklabels=True, yticklabels=True)
plt.xticks(ticks=np.arange(len(feature_cols)) + 0.5, labels=[f'ICA{i+1}' for i in range(corr_ica.shape[0])], rotation=45, ha='right')
plt.yticks(ticks=np.arange(len(feature_cols)) + 0.5, labels=[f'ICA{i+1}' for i in range(corr_ica.shape[0])], rotation=0)    
plt.title('Correlation Matrix of ICA Components')
plt.subplot(1, 3, 2)
sns.heatmap(corr_pca, annot=True, cmap='coolwarm', vmin=-1, vmax=1, xticklabels=True, yticklabels=True)
plt.xticks(ticks=np.arange(len(feature_cols)) + 0.5, labels=[f'PCA{i+1}' for i in range(corr_pca.shape[0])], rotation=45, ha='right')
plt.yticks(ticks=np.arange(len(feature_cols)) + 0.5, labels=[f'PCA{i+1}' for i in range(corr_pca.shape[0])], rotation=0)
plt.title('Correlation Matrix of PCA Components')
plt.subplot(1, 3, 3)
sns.heatmap(corr_orig, annot=True, cmap='coolwarm', vmin=-1, vmax=1, xticklabels=True, yticklabels=True)
if len(feature_cols) == 4:
    plt.xticks(ticks=np.arange(len(feature_cols)) + 0.5, labels=['$\\log M_{\\rm h}$', 'c', '$\\lambda$', '$a_{1/2}$'], rotation=45, ha='right')
    plt.yticks(ticks=np.arange(len(feature_cols)) + 0.5, labels=['$\\log M_{\\rm h}$', 'c', '$\\lambda$', '$a_{1/2}$'], rotation=0)
plt.title('Correlation Matrix of Original Properties')
plt.tight_layout()
plt.show()


In [ ]:
# Pairwise MI for each method
def pairwise_mi(X, random_state=31579):
    n = X.shape[1]
    mi = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j:
                mi[i, j] = mutual_info_regression(
                    X[:, j:j+1], X[:, i], random_state=random_state, n_jobs=-1
                )[0]
    return mi

idx = np.random.choice(range(reduced_ica.shape[0]), 10000, replace=False)
sample_ica = reduced_ica[idx]
idx = np.random.choice(range(reduced_pca.shape[0]), 10000, replace=False)
sample_pca = reduced_pca[idx]
idx = np.random.choice(range(train_clean.shape[0]), 10000, replace=False)
sample_orig = train_clean.iloc[idx]
sample_orig = sample_orig[feature_cols].to_numpy()

mi_ica = pairwise_mi(sample_ica)
mi_pca = pairwise_mi(sample_pca)
mi_orig = pairwise_mi(sample_orig)

# Too slow! TODO check this somehow though...
#mi_ica = pairwise_mi(all_ica_full)
#mi_pca = pairwise_mi(all_pca_full)
#mi_orig = pairwise_mi(all_clean[feature_cols].to_numpy())

vmax = max(mi_ica.max(), mi_pca.max(), mi_orig.max())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
labels_ica = [f'IC{i+1}' for i in range(n_features)]
labels_pca = [f'PC{i+1}' for i in range(n_features)]
labels_orig = [f'{col}' for col in feature_cols]
for ax, mi, labels, title in zip(axes, [mi_ica, mi_pca, mi_orig], [labels_ica, labels_pca, labels_orig], ['ICA', 'PCA', 'Original']):
    im = ax.imshow(mi, cmap='Oranges', vmin=0, vmax=vmax)
    ax.set_xticks(range(n_features)); ax.set_xticklabels(labels)
    ax.set_yticks(range(n_features)); ax.set_yticklabels(labels)
    for r in range(n_features):
        for c in range(n_features):
            ax.text(c, r, f'{mi[r,c]:.3f}', ha='center', va='center', fontsize=16)
    ax.set_title(f'{title} MI')

fig.colorbar(im, ax=axes[2], location='right')
plt.tight_layout()

print(f"ICA mean off-diagonal MI: {mi_ica[mi_ica > 0].mean():.4f}")
print(f"PCA mean off-diagonal MI: {mi_pca[mi_pca > 0].mean():.4f}")
print(f"Original mean off-diagonal MI: {mi_orig[mi_orig > 0].mean():.4f}")

The residual IC2–IC3 dependence you saw (KS=0.110) is physically expected: concentration and formation epoch are genuinely coupled (older halos are more concentrated — this is the physical origin of the mass-concentration relation). This is a real nonlinear correlation in halo physics that no linear transform can fully remove. It corresponds to what people call assembly bias.

In [ ]:
# Print number of halos
print(f"Total halos: {len(reduced_ica):,}")
print(f"Total halos: {len(all_with_ica):,}")

### Save Model


In [ ]:
# Save off the model and scaler for later use
if n_features == 2: 
    pkl_file = HALO_2P_MODEL_FILE
    txt_file = HALO_2P_MODEL_TEXT_FILE
elif n_features == 4:
    pkl_file = HALO_ICA_MODEL_FILE
    txt_file = HALO_ICA_MODEL_TEXT_FILE
else:
    raise ValueError(f"Unexpected number of features: {n_features}")

joblib.dump((ica, scaler, feature_cols), pkl_file)

# Export ICA model to plain text for C++ consumption
# Format: header lines with # comments, then data blocks
n_comp, n_feat = ica.components_.shape

with open(txt_file, 'w') as f:
    f.write(f"# Halo ICA model for C++ use\n")
    f.write(f"# features: {' '.join(feature_cols)}\n")
    f.write(f"# n_features: {n_feat}\n")
    f.write(f"# n_components: {n_comp}\n")
    f.write(f"\n# scaler_mean ({n_feat})\n")
    np.savetxt(f, scaler.mean_.reshape(1, -1), fmt='%.10e')
    f.write(f"\n# scaler_scale ({n_feat})\n")
    np.savetxt(f, scaler.scale_.reshape(1, -1), fmt='%.10e')
    f.write(f"\n# ica_mean_in_scaled_space ({n_feat})\n")
    np.savetxt(f, ica.mean_.reshape(1, -1), fmt='%.10e')
    f.write(f"\n# ica_components ({n_comp} x {n_feat})  — row i is eigenvector i\n")
    np.savetxt(f, ica.components_, fmt='%.10e')
    f.write(f"\n# ica_mixing_matrix ({n_feat} x {n_comp})  — for inverse transform\n")
    np.savetxt(f, ica.mixing_, fmt='%.10e')  # shape (n_feat, n_comp)

print(f"Wrote C++ ICA model to {txt_file}")

In [ ]:
# FOR GROUP FINDER ONLY
# LATSHAM does not need these since it uses the halos from the sim directly.

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ax = axes[0]
ax2 = axes[1]
ax.set_xlabel('ICAx Value')
ax2.set_xlabel('ICAx Value (zoom)')
ax.set_yscale('log')
ax2.set_yscale('log')
ax.set_ylabel('Density [1/(Mpc/h)^3]')
ax2.set_ylabel('Density [1/(Mpc/h)^3]')
ax.grid(True, ls="--", lw=0.5, alpha=0.5)
ax2.grid(True, ls="--", lw=0.5, alpha=0.5)

density_files = [HALO_PCA1_DENSITY_FUNC_FILE, HALO_PCA2_DENSITY_FUNC_FILE,
                 HALO_PCA3_DENSITY_FUNC_FILE, HALO_PCA4_DENSITY_FUNC_FILE]

for i in range(complete_sample.shape[1]):
    colname = f'ICA{i+1}'
    data = complete_sample[colname].values

    b = make_adaptive_density_bins(data, 700, 30, 0.35, 3e-8)
    print(f"{colname}: data [{data.min():.2f}, {data.max():.2f}]  "
          f"bins [{b[0]:.2f}, {b[-1]:.2f}]  n_bins={len(b)-1}")

    dICA = np.diff(b)
    counts, _ = np.histogram(data, bins=b)
    centers = 0.5 * (b[:-1] + b[1:])
    density = counts / dICA / BOX_VOLUME
    output_data = np.column_stack((centers, density))

    ax.plot(centers, density, drawstyle='steps-mid', label=colname, color=pp.get_color(i))
    axes[1].plot(centers, density, drawstyle='steps-mid', label=colname, color=pp.get_color(i))
    #np.savetxt(density_files[i], output_data)

ax.legend()
axes[1].set_xlim(-4.5, 3.5)
axes[1].set_ylim(1e-3, 0.9)
plt.tight_layout()
plt.show()

### Tests

In [ ]:
testpoints = 10000
sample = all_clean.iloc[:testpoints]
there = halo_original_to_latent(sample[feature_cols].to_numpy())
latent_cols = [f'ICA{i+1}' for i in range(n_features)]
back = halo_latent_to_original(there)
tol = 1e-5 # more than really needed
assert np.allclose(sample[feature_cols], back, atol=tol), "Roundtrip ICA transformation did not recover original values within tolerance!"

In [ ]:
# Used in a C++ test to verify the model is being loaded and applied correctly there too.
print(all_with_ica.iloc[0])